# ValidEval GSM8K V4 Production Kaggle Runner

Project goal: produce real GSM8K model-output rows, a correctness matrix, and a strict import ZIP for ValidEval V4

Expected runtime: smoke 5-20 min; small panel 2-6 hr; medium panel 6-12 hr depending on accelerator and model availability

Accelerator requirement: GPU strongly recommended; CPU is smoke/preflight only. Use Kaggle T4/P100/V100/A100 when running real model inference.

Output ZIP name: `valideval_outputs.zip`.

Resume behavior: the notebook writes one partial shard per model/task and skips shards whose normalized partial output already exists. It updates `model_status.csv` and `failed_models.csv` after every shard.

Evidence boundary: this notebook can produce importable benchmark outputs when run externally. Until the downloaded ZIP is imported locally with `python3 -m valideval import-kaggle-outputs --strict`, it is not ValidEval evidence. It does not claim GSM8K, BBH, TruthfulQA, human-label, external-label, or cross-benchmark results by itself.


## Runtime And Panel Modes

| Mode | Models | Items/tasks | Expected runtime | Use when |
| --- | --- | --- | --- | --- |
| `smoke` | 1 tiny fallback | 5-20 examples | 5-20 min | preflight/resume/schema test |
| `small` | 3 open models | full selected task unless `VALIDEVAL_LIMIT` set | 2-6 hr | first evidence-producing run |
| `medium` | 5 open models | full selected task unless `VALIDEVAL_LIMIT` set | 6-12 hr | stronger panel after small passes |


In [ ]:
import importlib.util
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path

BENCHMARK_DEFAULT = "gsm8k"
TASKS_DEFAULT = "gsm8k"
OUTPUT_ZIP_NAME = "valideval_outputs.zip"
OUTPUT_DIR = Path(os.environ.get("VALIDEVAL_OUTPUT_DIR", "/kaggle/working/valideval_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = OUTPUT_DIR / "run_log.txt"


def log(message):
    stamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    line = f"{stamp} {message}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def command_text(cmd):
    try:
        return subprocess.run(
            cmd, check=False, capture_output=True, text=True, timeout=20
        ).stdout.strip()
    except Exception as exc:
        return f"unavailable: {exc}"


def internet_available():
    try:
        socket.create_connection(("huggingface.co", 443), timeout=5).close()
        return True
    except OSError:
        return False


def path_writable(path):
    try:
        probe = path / ".write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except Exception:
        return False


packages = {
    name: importlib.util.find_spec(name) is not None
    for name in ["numpy", "pandas", "torch", "transformers", "datasets", "lm_eval", "yaml"]
}
task_paths = {
    "src/valideval": Path("src/valideval").exists(),
    "kaggle_general": Path("kaggle_general").exists(),
    "kaggle_gsm8k": Path("kaggle_gsm8k").exists(),
    "kaggle_third_benchmark": Path("kaggle_third_benchmark").exists(),
}
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "cwd": str(Path.cwd()),
    "gpu_nvidia_smi": command_text(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
    ),
    "disk_free_gb": round(shutil.disk_usage(str(OUTPUT_DIR)).free / 1e9, 2),
    "packages": packages,
    "internet_available": internet_available(),
    "task_availability": task_paths,
    "output_dir": str(OUTPUT_DIR),
    "output_dir_writable": path_writable(OUTPUT_DIR),
}
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True), encoding="utf-8"
)
log("preflight complete")
print(json.dumps(environment, indent=2, sort_keys=True))
if not environment["output_dir_writable"]:
    raise RuntimeError("Output directory is not writable")

In [ ]:
import json
import os
import re
from pathlib import Path

try:
    import yaml
except Exception:
    yaml = None

BENCHMARK = os.environ.get("VALIDEVAL_BENCHMARK", BENCHMARK_DEFAULT).strip().lower()
TASKS = [
    part.strip()
    for part in os.environ.get("VALIDEVAL_TASKS", TASKS_DEFAULT).split(",")
    if part.strip()
]
RUN_MODE = os.environ.get("VALIDEVAL_RUN_MODE", "smoke").strip().lower()
TASK_SUBSET = [
    part.strip() for part in os.environ.get("VALIDEVAL_TASK_SUBSET", "").split(",") if part.strip()
]
if TASK_SUBSET:
    TASKS = TASK_SUBSET
LIMIT = os.environ.get("VALIDEVAL_LIMIT", "20" if RUN_MODE == "smoke" else "")
EXECUTE_LM_EVAL = os.environ.get("EXECUTE_LM_EVAL", "0") == "1"
REQUIRE_PREDICTIONS = os.environ.get("REQUIRE_PREDICTIONS", "1") == "1"
DEVICE = os.environ.get("VALIDEVAL_DEVICE", "cuda:0")
BATCH_SIZE = os.environ.get("VALIDEVAL_BATCH_SIZE", "auto")


def load_models(path, fallback):
    if yaml is None or not Path(path).exists():
        return fallback
    data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
    return list(data.get("models") or fallback)


MODEL_PANELS = {
    "smoke": [os.environ.get("VALIDEVAL_SMOKE_MODEL", "sshleifer/tiny-gpt2")],
    "small": load_models(
        "kaggle_gsm8k/gsm8k_models_small.yaml",
        [
            "Qwen/Qwen2.5-0.5B-Instruct",
            "Qwen/Qwen2.5-1.5B-Instruct",
            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        ],
    ),
    "medium": load_models(
        "kaggle_gsm8k/gsm8k_models_medium.yaml",
        [
            "Qwen/Qwen2.5-0.5B-Instruct",
            "Qwen/Qwen2.5-1.5B-Instruct",
            "Qwen/Qwen2.5-3B-Instruct",
            "microsoft/Phi-3-mini-4k-instruct",
            "google/gemma-2-2b-it",
        ],
    ),
    "subset": load_models(
        "kaggle_general/model_panels/small_open.yaml",
        [
            "Qwen/Qwen2.5-0.5B-Instruct",
            "Qwen/Qwen2.5-1.5B-Instruct",
            "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        ],
    ),
    "full": load_models(
        "kaggle_gsm8k/gsm8k_models_medium.yaml",
        [
            "Qwen/Qwen2.5-0.5B-Instruct",
            "Qwen/Qwen2.5-1.5B-Instruct",
            "Qwen/Qwen2.5-3B-Instruct",
            "microsoft/Phi-3-mini-4k-instruct",
            "google/gemma-2-2b-it",
        ],
    ),
}
MODELS = MODEL_PANELS.get(RUN_MODE, MODEL_PANELS["smoke"])
RUNTIME_TABLE = {
    "smoke": "5-30 min",
    "small": "2-6 hr",
    "medium": "6-12 hr",
    "subset": "1-4 hr",
    "full": "6-12 hr",
}
PLAN = {
    "benchmark": BENCHMARK,
    "tasks": TASKS,
    "run_mode": RUN_MODE,
    "models": MODELS,
    "limit": LIMIT or None,
    "execute_lm_eval": EXECUTE_LM_EVAL,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "expected_runtime": RUNTIME_TABLE.get(RUN_MODE, "unknown"),
    "output_zip_name": OUTPUT_ZIP_NAME,
}
(OUTPUT_DIR / "run_plan.json").write_text(
    json.dumps(PLAN, indent=2, sort_keys=True), encoding="utf-8"
)
print(json.dumps(PLAN, indent=2, sort_keys=True))

In [ ]:
import csv
import json
import shlex
from pathlib import Path

PARTIAL_DIR = OUTPUT_DIR / "partials"
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
STATUS_CSV = OUTPUT_DIR / "model_status.csv"
FAILED_CSV = OUTPUT_DIR / "failed_models.csv"

STATUS_FIELDS = ["benchmark", "task", "model_id", "status", "partial_predictions", "message"]
FAILED_FIELDS = ["benchmark", "task", "model_id", "error"]

if not STATUS_CSV.exists():
    with STATUS_CSV.open("w", encoding="utf-8", newline="") as handle:
        csv.DictWriter(handle, fieldnames=STATUS_FIELDS).writeheader()
if not FAILED_CSV.exists():
    with FAILED_CSV.open("w", encoding="utf-8", newline="") as handle:
        csv.DictWriter(handle, fieldnames=FAILED_FIELDS).writeheader()


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_") or "unnamed"


def append_csv(path, fields, row):
    with path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writerow({field: row.get(field, "") for field in fields})


def normalize_sample_record(record, *, benchmark, task, model_id, row_number):
    doc = record.get("doc") if isinstance(record.get("doc"), dict) else {}
    metrics = record.get("metrics") if isinstance(record.get("metrics"), dict) else {}
    item_id = (
        record.get("doc_id")
        or record.get("item_id")
        or doc.get("id")
        or doc.get("question_id")
        or f"{task}_{row_number}"
    )
    prediction = (
        record.get("filtered_resps")
        or record.get("resps")
        or record.get("prediction")
        or record.get("output")
        or ""
    )
    if isinstance(prediction, list):
        prediction = prediction[0] if prediction else ""
    if isinstance(prediction, list):
        prediction = prediction[0] if prediction else ""
    gold = (
        record.get("target")
        or doc.get("target")
        or doc.get("answer")
        or doc.get("gold")
        or record.get("gold")
        or ""
    )
    correct = record.get("exact_match")
    if correct is None:
        correct = record.get("acc")
    if correct is None:
        correct = metrics.get("exact_match") or metrics.get("acc")
    if isinstance(correct, list):
        correct = correct[0] if correct else None
    if correct is None:
        correct = str(prediction).strip().lower() == str(gold).strip().lower()
    return {
        "benchmark": benchmark,
        "subset": task,
        "model_id": model_id,
        "item_id": str(item_id),
        "prediction": str(prediction),
        "gold": str(gold),
        "correct": bool(float(correct))
        if isinstance(correct, (int, float, str))
        and str(correct).strip() in {"0", "1", "0.0", "1.0"}
        else bool(correct),
        "metadata": {"source": "lm_eval_kaggle", "task": task},
    }


def normalize_lm_eval_samples(shard_dir, *, benchmark, task, model_id):
    rows = []
    for sample_file in sorted(Path(shard_dir).rglob("*.jsonl")):
        if "normalized_predictions" in sample_file.name:
            continue
        with sample_file.open("r", encoding="utf-8") as handle:
            for row_number, line in enumerate(handle, start=1):
                if line.strip():
                    rows.append(
                        normalize_sample_record(
                            json.loads(line),
                            benchmark=benchmark,
                            task=task,
                            model_id=model_id,
                            row_number=row_number,
                        )
                    )
    out = Path(shard_dir) / "normalized_predictions.jsonl"
    if rows:
        out.write_text(
            "".join(json.dumps(row, sort_keys=True) + "\n" for row in rows), encoding="utf-8"
        )
    return out, len(rows)


for model_id in MODELS:
    for task in TASKS:
        shard_dir = PARTIAL_DIR / safe_name(BENCHMARK) / safe_name(task) / safe_name(model_id)
        normalized = shard_dir / "normalized_predictions.jsonl"
        if normalized.exists() and normalized.stat().st_size > 0:
            append_csv(
                STATUS_CSV,
                STATUS_FIELDS,
                {
                    "benchmark": BENCHMARK,
                    "task": task,
                    "model_id": model_id,
                    "status": "skipped_completed",
                    "partial_predictions": str(normalized),
                    "message": "resume hit",
                },
            )
            log(f"skip completed shard {model_id} {task}")
            continue
        shard_dir.mkdir(parents=True, exist_ok=True)
        if not EXECUTE_LM_EVAL:
            append_csv(
                STATUS_CSV,
                STATUS_FIELDS,
                {
                    "benchmark": BENCHMARK,
                    "task": task,
                    "model_id": model_id,
                    "status": "pending_execute_lm_eval_false",
                    "partial_predictions": "",
                    "message": "set EXECUTE_LM_EVAL=1 for real run",
                },
            )
            log(f"pending shard {model_id} {task}; EXECUTE_LM_EVAL=0")
            continue
        cmd = [
            sys.executable,
            "-m",
            "lm_eval",
            "--model",
            "hf",
            "--model_args",
            f"pretrained={model_id},trust_remote_code=True",
            "--tasks",
            task,
            "--device",
            DEVICE,
            "--batch_size",
            BATCH_SIZE,
            "--output_path",
            str(shard_dir),
            "--log_samples",
        ]
        if LIMIT:
            cmd.extend(["--limit", str(LIMIT)])
        log("running " + " ".join(shlex.quote(part) for part in cmd))
        try:
            subprocess.run(cmd, check=True)
            normalized, row_count = normalize_lm_eval_samples(
                shard_dir, benchmark=BENCHMARK, task=task, model_id=model_id
            )
            status = "completed" if row_count else "failed_no_sample_rows"
            append_csv(
                STATUS_CSV,
                STATUS_FIELDS,
                {
                    "benchmark": BENCHMARK,
                    "task": task,
                    "model_id": model_id,
                    "status": status,
                    "partial_predictions": str(normalized) if row_count else "",
                    "message": f"rows={row_count}",
                },
            )
            if not row_count:
                append_csv(
                    FAILED_CSV,
                    FAILED_FIELDS,
                    {
                        "benchmark": BENCHMARK,
                        "task": task,
                        "model_id": model_id,
                        "error": "lm_eval finished but no sample rows were normalized",
                    },
                )
        except Exception as exc:
            append_csv(
                STATUS_CSV,
                STATUS_FIELDS,
                {
                    "benchmark": BENCHMARK,
                    "task": task,
                    "model_id": model_id,
                    "status": "failed",
                    "partial_predictions": "",
                    "message": str(exc),
                },
            )
            append_csv(
                FAILED_CSV,
                FAILED_FIELDS,
                {"benchmark": BENCHMARK, "task": task, "model_id": model_id, "error": str(exc)},
            )
            log(f"failed shard {model_id} {task}: {exc}")

In [ ]:
import json
from collections import Counter

import pandas as pd

prediction_rows = []
for partial in sorted(PARTIAL_DIR.rglob("normalized_predictions.jsonl")):
    with partial.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                prediction_rows.append(json.loads(line))

PREDICTIONS = OUTPUT_DIR / "predictions.jsonl"
MATRIX = OUTPUT_DIR / "matrix.csv"
if not prediction_rows:
    log("no normalized prediction rows found")
    if REQUIRE_PREDICTIONS:
        raise RuntimeError(
            "No prediction rows were produced. Do not package fake outputs; run with EXECUTE_LM_EVAL=1 or inspect failed_models.csv."
        )
else:
    PREDICTIONS.write_text(
        "".join(json.dumps(row, sort_keys=True) + "\n" for row in prediction_rows), encoding="utf-8"
    )
    frame = pd.DataFrame(
        [
            {
                "model_id": row["model_id"],
                "item_key": f"{row.get('subset', 'default')}::{row['item_id']}",
                "correct": float(bool(row["correct"])),
            }
            for row in prediction_rows
        ]
    )
    duplicates = [
        key
        for key, count in Counter(zip(frame["model_id"], frame["item_key"], strict=False)).items()
        if count > 1
    ]
    if duplicates:
        raise RuntimeError(f"Duplicate model/item rows before matrix build: {duplicates[:5]}")
    matrix = (
        frame.pivot(index="model_id", columns="item_key", values="correct")
        .sort_index()
        .sort_index(axis=1)
    )
    matrix.to_csv(MATRIX)
    log(
        f"wrote predictions={PREDICTIONS} rows={len(prediction_rows)} matrix={MATRIX} shape={matrix.shape}"
    )

In [ ]:
import json
from collections import Counter
from pathlib import Path

required = {"benchmark", "model_id", "item_id", "prediction", "gold", "correct"}
rows = []
with PREDICTIONS.open("r", encoding="utf-8") as handle:
    for idx, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        missing = required - set(row)
        if missing:
            raise RuntimeError(f"row {idx} missing required fields: {sorted(missing)}")
        if row["prediction"] in (None, ""):
            raise RuntimeError(f"row {idx} has missing prediction")
        if not isinstance(row["item_id"], str) or not row["item_id"].strip():
            raise RuntimeError(f"row {idx} has unstable item_id")
        if not isinstance(row["correct"], bool):
            raise RuntimeError(f"row {idx} has non-boolean correct field")
        rows.append(row)
keys = Counter((row["model_id"], row.get("subset", "default"), row["item_id"]) for row in rows)
dups = [key for key, count in keys.items() if count > 1]
if dups:
    raise RuntimeError(f"duplicate model/subset/item rows: {dups[:5]}")
matrix = pd.read_csv(MATRIX, index_col=0)
validation = {
    "row_count": len(rows),
    "model_count": len({row["model_id"] for row in rows}),
    "item_count": len({(row.get("subset", "default"), row["item_id"]) for row in rows}),
    "duplicate_model_item_rows": len(dups),
    "missing_prediction_count": 0,
    "correctness_field_check": "pass",
    "stable_item_id_check": "pass",
    "matrix_shape": list(matrix.shape),
}
(OUTPUT_DIR / "validation.json").write_text(
    json.dumps(validation, indent=2, sort_keys=True), encoding="utf-8"
)
print(json.dumps(validation, indent=2, sort_keys=True))

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

required_files = [
    "predictions.jsonl",
    "matrix.csv",
    "manifest.json",
    "model_status.csv",
    "failed_models.csv",
    "run_log.txt",
    "environment.json",
]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "schema_version": "v4_kaggle_output",
    "benchmark": BENCHMARK,
    "tasks": TASKS,
    "run_mode": RUN_MODE,
    "models": MODELS,
    "evidence_boundary": "import_required_before_local_claims",
    "files": {},
}
for name in required_files:
    path = OUTPUT_DIR / name
    if name != "manifest.json" and not path.exists():
        raise RuntimeError(f"Required package file is missing: {path}")
    if path.exists() and name != "manifest.json":
        manifest["files"][name] = {"bytes": path.stat().st_size, "sha256": sha256(path)}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
)
manifest["files"]["manifest.json"] = {
    "bytes": (OUTPUT_DIR / "manifest.json").stat().st_size,
    "sha256": sha256(OUTPUT_DIR / "manifest.json"),
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
)
zip_path = Path("/kaggle/working") / OUTPUT_ZIP_NAME
if not Path("/kaggle/working").exists():
    zip_path = Path.cwd() / OUTPUT_ZIP_NAME
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(OUTPUT_DIR / name, name)
print(
    json.dumps(
        {"zip_path": str(zip_path), "sha256": sha256(zip_path), "files": required_files},
        indent=2,
        sort_keys=True,
    )
)

## Resume Notes

Re-run the notebook with the same output directory. Completed shards are skipped from `partials/`; failed shards remain in `failed_models.csv` for inspection.


## Final Download And Local Import

Download `valideval_outputs.zip` from `/kaggle/working/`.

Place it locally under:

```text
kaggle_outputs/gsm8k/
```

Then run from the ValidEval repo:

```bash
python3 -m valideval import-kaggle-outputs --input-dir kaggle_outputs --output-root data/external/kaggle_imported --cache-root cache --results-root results --strict
```

After import, run the post-import router for each imported benchmark matrix. For GSM8K:

```bash
python3 -m valideval post-import-analysis --benchmark gsm8k --matrix cache/gsm8k/wide/matrix.csv --predictions cache/gsm8k/wide/predictions.jsonl --output results/gsm8k --execute
```
